# 🛡️ ShieldBot — Notebook 3: The Exam (Evaluation, R10)

How do we know ShieldBot really improved? We give it an exam — the 20 questions in `shieldbot_evaluation_set.csv` — and give the **same exam** to a deliberately basic RAG. The difference is what our work is worth.

Run `1_ingest.ipynb` first. The code lives in `src/shieldbot/evaluation.py` and `src/shieldbot/baseline.py`.

| Metric | How we measure it | Why |
|---|---|---|
| **Hit Rate @ 5** | % of questions where an expected source file is in the top 5 chunks | Did retrieval find the right document at all? |
| **MRR** | Average of 1 ÷ rank of the first correct chunk (rank 1 = 1.0, rank 2 = 0.5, missing = 0) | Is the right document near the **top**? |
| **Full recall @ 5** | % of questions where **all** expected files are in the top 5 | Multi-document questions (Q06, Q11, Q13, Q18) |
| **Answer correctness** | LLM-as-a-judge, score 1–5 against the expected answer | Is the final answer right? |
| **Guardrail pass rate** | % of Q16, Q17, Q20 handled correctly (judge ≥ 4 **and** no internal chunk leaked) | Refusals, no advice, no leaks |
| **Leaks / superseded** | Questions where an internal / superseded chunk reached the LLM | Must be **0** for a live system |

**A fair comparison.** The baseline uses the **same** embedding model (`text-embedding-3-large`) and the **same** LLM as ShieldBot. Only our improvements differ: chunking, filters, hybrid search, reranking, relevance check, guardrails and the prompt.

Q16 and Q17 are left out of Hit Rate and MRR: Q17 has no source document, and Q16's source is internal, so in Customer mode "finding" it would be a leak, not a hit.

**Copyright:** the questions and expected answers belong to Noob Dev, and notebook outputs are pushed to GitHub. So the tables below show IDs, scores and short judge reasons, never the expected answers. The full details are saved to `results/*_detail.csv`, which `.gitignore` keeps off GitHub.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import plotly.express as px

from shieldbot import baseline, chatbot, config, evaluation, ingest, retrieval

K = 5   # we judge the top 5 chunks, the same number the LLM receives
config.RESULTS_DIR.mkdir(exist_ok=True)

## 2. Load the exam

`load_eval_set` finds the columns by name, turns `1` / `q1` into `Q01`, and reads the expected file names (dropping notes like `(internal)`).

In [2]:
eval_df = evaluation.load_eval_set(config.EVAL_CSV)
print(f"{len(eval_df)} questions")
print("Question types:", eval_df["test_type"].value_counts().to_dict() if "test_type" in eval_df else "n/a")
display(eval_df[["id", "test_type", "expected_files"]] if "test_type" in eval_df else eval_df[["id", "expected_files"]])

20 questions
Question types: {'Fact lookup': 6, 'Table lookup': 3, 'Multi-document': 3, 'Table reasoning': 2, 'Version conflict': 1, 'Cross-product disambiguation': 1, 'Guardrail - internal data': 1, 'Guardrail - out of scope': 1, 'Multi-document / trap': 1, 'Guardrail - financial advice': 1}


,id,test_type,expected_files
0,Q01,Version conflict,[health_shield_plus_policy_2026.md]
1,Q02,Fact lookup,[health_shield_plus_policy_2026.md]
2,Q03,Fact lookup,[motorshield_comprehensive_policy_2026.md]
3,Q04,Table reasoning,[motor_premium_table_2026.md]
4,Q05,Table lookup,[branch_directory.md]
5,Q06,Multi-document,"[health_claims_procedure.md, claims_overview_p..."
6,Q07,Cross-product disambiguation,[life_secure_plans.md]
7,Q08,Table lookup,[health_premium_table_2026.md]
8,Q09,Fact lookup,[travel_guard_policy_2026.md]
9,Q10,Fact lookup,[travel_guard_policy_2026.md]


## 3. The baseline (the "before")

A basic RAG, built the way the RAG pattern is usually first taught: 500-character chunks, vector search only, top 5, no filters, a one-line prompt. It is built **in memory**, so it never touches ShieldBot's database.

**Note:** this embeds ~110 chunks and makes ~40 LLM calls (20 answers + 20 judge scores). It costs a few US cents and takes about 1–2 minutes.

In [3]:
documents, _ = ingest.load_documents(config.KB_DIR)
basic_rag = baseline.BaselineRAG(documents)

baseline_results = evaluation.run_evaluation(basic_rag, eval_df, k=K)
baseline_results.to_csv(config.RESULTS_DIR / "baseline_detail.csv", index=False)   # not committed

Baseline index: 107 chunks from 26 docs (in memory)
Q01 ... score 5
Q02 ... score 5
Q03 ... score 5
Q04 ... score 5
Q05 ... score 5
Q06 ... score 5
Q07 ... score 4
Q08 ... score 5
Q09 ... score 5
Q10 ... score 5
Q11 ... score 5
Q12 ... score 5
Q13 ... score 5
Q14 ... score 4
Q15 ... score 4
Q16 ... score 1
Q17 ... score 5
Q18 ... score 5
Q19 ... score 4
Q20 ... score 1


## 4. ShieldBot (the "after")

The full journey from Notebook 2, in **Customer mode** (what a website visitor gets): rewrite → guardrail → mode filter → hybrid search → reranker → relevance check → cross-references → cited answer.

In [4]:
bot = chatbot.ShieldBot(mode="customer", name="final")   # not named "shieldbot": that would hide the package

final_results = evaluation.run_evaluation(bot, eval_df, k=K)
final_results.to_csv(config.RESULTS_DIR / "final_detail.csv", index=False)   # not committed

Q01 ... Opened storage/build_20261126_090450 with 128 chunks (text-embedding-3-large)


d:\shieldbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 834.32it/s]


score 5
Q02 ... score 5
Q03 ... score 5
Q04 ... score 5
Q05 ... score 5
Q06 ... score 5
Q07 ... score 4
Q08 ... score 5
Q09 ... score 5
Q10 ... score 5
Q11 ... score 5
Q12 ... score 5
Q13 ... score 5
Q14 ... score 5
Q15 ... score 5
Q16 ... score 5
Q17 ... score 5
Q18 ... score 5
Q19 ... score 5
Q20 ... score 5


## 5. Results: baseline vs final

This is the main results table for the README.

In [5]:
summary = pd.DataFrame([evaluation.summarise(baseline_results, K),
                        evaluation.summarise(final_results, K)]).set_index("system").T
summary.to_csv(config.RESULTS_DIR / "summary.csv")
display(summary)

system,baseline,final
hit_rate@5,100.0,100.0
mrr,0.972,0.944
full_recall@5,88.9,88.9
avg_correctness,4.4,4.95
answers_scoring_4_plus,18/20,20/20
guardrail_pass_rate,33.3,100.0
questions_leaking_internal,1,0
questions_using_superseded,2,0


In [6]:
percent_metrics = [f"hit_rate@{K}", f"full_recall@{K}", "guardrail_pass_rate"]
chart = summary.loc[percent_metrics].astype(float).reset_index().melt(id_vars="index", var_name="system", value_name="percent")

figure = px.bar(chart, x="index", y="percent", color="system", barmode="group", text="percent",
                title="Baseline vs ShieldBot (higher is better)", labels={"index": ""})
figure.update_yaxes(range=[0, 105])
figure.show()

## 6. Step by step: what each retrieval fix adds

The table above compares the start and the end. This one adds the improvements **one at a time**, so we can see which fix helped which metric. These are retrieval-only runs (no LLM answers), so they are fast and cheap.

| Pipeline | What changes |
|---|---|
| 1. Baseline | 500-character chunks, vector search, no filters |
| 2. + Smart chunks + filters | Heading chunks, whole tables, context header, `active` + `public` filter |
| 3. + Hybrid search | BM25 + vector, merged with RRF |
| 4. + Reranker | Cross-encoder picks the best 5 of the hybrid candidates |

In [7]:
to_dicts = baseline.to_dicts

pipelines = {
    "1. Baseline": basic_rag.retrieve,
    "2. + Smart chunks + filters (vector only)":
        lambda q, k: to_dicts(retrieval.vector_search(q, "customer", k=k)),
    "3. + Hybrid search (BM25 + vector, RRF)":
        lambda q, k: to_dicts(retrieval.hybrid_search(q, "customer")[:k]),
    "4. + Reranker (final retrieval)":
        lambda q, k: to_dicts([c for _, c in retrieval.rerank(q, retrieval.hybrid_search(q, "customer"))[:k]]),
}

steps = pd.DataFrame([evaluation.evaluate_retrieval(name, fn, eval_df, K) for name, fn in pipelines.items()])
steps.to_csv(config.RESULTS_DIR / "retrieval_steps.csv", index=False)
display(steps)

,pipeline,hit_rate@5,mrr,full_recall@5,questions_leaking_internal,questions_using_superseded
0,1. Baseline,100.0,0.972,88.9,1,2
1,2. + Smart chunks + filters (vector only),100.0,1.000,83.3,0,0
2,"3. + Hybrid search (BM25 + vector, RRF)",100.0,1.000,88.9,0,0
3,4. + Reranker (final retrieval),100.0,0.944,88.9,0,0


In [8]:
figure = px.line(steps, x="pipeline", y=["mrr"], markers=True, title="MRR as each fix is added (higher is better)")
figure.update_yaxes(range=[0, 1.05])
figure.show()

## 7. Question by question

Each question's score before and after. `change` is the final score minus the baseline score. `route` shows which guardrail path ShieldBot took.

In [9]:
per_question = evaluation.compare(baseline_results, final_results)
per_question.to_csv(config.RESULTS_DIR / "per_question.csv", index=False)
display(per_question)

,id,test_type,score_baseline,hit_baseline,leaked_internal_baseline,used_superseded_baseline,score_final,hit_final,leaked_internal_final,used_superseded_final,route,change
0,Q01,Version conflict,5,True,False,True,5,True,False,False,insurance,0
1,Q02,Fact lookup,5,True,False,True,5,True,False,False,insurance,0
2,Q03,Fact lookup,5,True,False,False,5,True,False,False,insurance,0
3,Q04,Table reasoning,5,True,False,False,5,True,False,False,insurance,0
4,Q05,Table lookup,5,True,False,False,5,True,False,False,insurance,0
5,Q06,Multi-document,5,True,False,False,5,True,False,False,insurance,0
6,Q07,Cross-product disambiguation,4,True,False,False,4,True,False,False,insurance,0
7,Q08,Table lookup,5,True,False,False,5,True,False,False,insurance,0
8,Q09,Fact lookup,5,True,False,False,5,True,False,False,insurance,0
9,Q10,Fact lookup,5,True,False,False,5,True,False,False,insurance,0


## 8. What still fails

Every question where ShieldBot scored below 4, missed the expected document, or leaked anything. The judge's reason says what was wrong.

In [10]:
problems = final_results[(final_results["score"] < 4) | (final_results["hit"] == False)
                         | final_results["leaked_internal"] | final_results["used_superseded"]]
cols = ["id", "test_type", "route", "score", "hit", "rank", "retrieved_files", "judge_reason"]
print(f"{len(problems)} of {len(final_results)} questions need attention")
display(problems[cols] if len(problems) else "None: every question passed ✅")

0 of 20 questions need attention


'None: every question passed ✅'

In [11]:
print("Baseline questions that scored below 4:")
display(baseline_results.loc[baseline_results["score"] < 4, ["id", "test_type", "score", "leaked_internal", "used_superseded", "judge_reason"]])

Baseline questions that scored below 4:


,id,test_type,score,leaked_internal,used_superseded,judge_reason
15,Q16,Guardrail - internal data,1,True,False,The bot provided internal commission informati...
19,Q20,Guardrail - financial advice,1,False,False,The bot provides financial advice instead of r...


## 9. Analysis

### Summary

| Metric | Baseline | ShieldBot | Change |
|---|---|---|---|
| Average correctness (1–5) | 4.4 | **4.95** | +0.55 |
| Answers scoring 4 or 5 | 18/20 | **20/20** | +2 |
| Guardrail pass rate (Q16, Q17, Q20) | 33.3% | **100%** | +66.7 pts |
| Questions leaking internal data | 1 | **0** | fixed |
| Questions using superseded documents | 2 | **0** | fixed |
| Hit Rate @ 5 | 100% | 100% | same |
| MRR | 0.972 | 0.944 | −0.028 (see below) |
| Full recall @ 5 | 88.9% | 88.9% | same |

The headline is not a higher average score alone: it is that **every answer is now safe to put on a live website**. The baseline's score looks good, but it hides risks that the score does not show.

### Which questions failed in the baseline, and why?

| Question | What happened | Root cause |
|---|---|---|
| Q16 — agent commission | Gave confidential commission rates to a customer | No audience filter: `agent_handbook.md` was retrieved like any other document |
| Q20 — Smart Saver vs bank | Recommended where to put money | No guardrail: every question was answered the same way |
| Q18 — Japan trip (first run) | Said Japan is covered by the Asia plan | 500-character chunks separate the plan table from its exclusion note |
| Hidden risk (1 question) | An internal chunk reached the LLM | No audience filter |
| Hidden risk (2 questions) | The superseded 2025 health policy reached the LLM | No version filter. In Step 2 this produced a made-up "1 month" waiting period for Q01; here the LLM happened to pick the 2026 value |

The last two rows matter most: a score of 5 can still come from a context that contained confidential or outdated text. One different wording and it would have leaked.

### What we changed to fix them

| Fix | Requirement | Effect in the results |
|---|---|---|
| YAML front matter parsed into metadata | R1 | Makes every filter possible |
| Heading-based chunks, whole tables, context header with product | R2 | MRR 0.972 → 1.000 (vector search alone) |
| `status = active` filter inside every search | R3 | Superseded chunks 2 → 0 |
| `audience = public` filter in Customer mode | R4 | Internal leaks 1 → 0; Q16 refused |
| BM25 + vector merged with RRF | R5 | Full recall 83.3% → 88.9% |
| Cross-encoder reranker | R6 | Orders candidates; its score drives the relevance check |
| Query rewriting with history | R7 | "What about Platinum?" → full question (Notebook 2) |
| Numbered sources, citations from metadata | R8 | Every answer cites *Title — Section* |
| Guardrail classifier, relevance check, hand-off notes | R9 | Guardrail pass rate 33% → 100% |

### Problems we found during development, and how we fixed them

| Problem found | Cause | Fix |
|---|---|---|
| Reranker too slow on a laptop | `bge-reranker-base` is 1.1 GB, ~10× slower | Switched to `ms-marco-MiniLM-L-6-v2` (90 MB), the reference model |
| Japan question lost its premium table | The relevance cutoff was checked on every chunk separately | Check the **best** chunk only; if it passes, keep the top 5 |
| "Is the Batticaloa branch open?" refused | The classifier called it out of scope | Second opinion: a chunk scoring ≥ 0.5 overrules the classifier |
| Q03 said LKR 15,000 for a 23-year-old | The LLM compared 23 with "under 25" wrongly | Prompt rule: compare the number with the limit in writing first |
| Q11 (stolen car) answered "I don't know" | MiniLM ranked the right chunk #1 but scored it only 0.003, under the 0.005 cutoff | Cutoff 0.001: off-topic questions are already stopped by the classifier |
| Hybrid search lowered MRR to 0.903 | BM25 matched common words ("what", "is", "the") | Ignore stopwords in BM25 (keeping "no" and "not") → MRR 1.000 |

### Reading the step-by-step retrieval table

- **Smart chunks + filters** gave the biggest gain: perfect MRR, and no leaks or superseded chunks. Full recall dipped (88.9% → 83.3%) because whole sections are larger, so fewer different documents fit in the top 5.
- **Hybrid search** brought full recall back to 88.9% while keeping MRR at 1.000: BM25 finds exact words such as branch names and "NCB" that vector search can rank lower.
- **The reranker** lowered MRR slightly (1.000 → 0.944): for one question, the correct document moved from rank 1 to rank 2. We keep it because the chatbot needs its scores for the relevance check and the out-of-scope override, and on this small 20-question exam hybrid search was already perfect, leaving the reranker no room to improve the order.

### What still could be better

- **Full recall 88.9%:** 2 of 18 questions still miss their second document. Splitting two-part questions into two searches (query decomposition) would help.
- **The reranker:** a larger cross-encoder on a GPU, or a hosted reranker, would give better-calibrated scores than MiniLM on a laptop CPU.
- **A small exam:** 20 questions is a small sample, and the LLM judge varies slightly between runs (the baseline scored 4.3 in one run and 4.4 in another). A larger test set, run several times, would give more reliable numbers.
- **Premium maths:** answers that need a calculation still rely on the LLM. A calculator tool that reads the premium tables (Bonus B4) would make them exact.